# Orthogonal Fourier Epicycles: Error Analysis & Complex Chain Comparison

This notebook conducts a mathematical and empirical error analysis of the **Orthogonal Fourier Epicycles** method for parametric boundary curves, using India's national boundary as the benchmark geometry.

### Core Questions Explored:
1. **Harmonic Convergence Rate**: How rapidly does the mean distance error decay as harmonic order $N$ spans from $1$ to $2000$ on a log-log scale?
2. **Visual Fidelity**: Side-by-side geometric reconstructions across $N \in \{1, 5, 20, 100, 1000\}$.
3. **Orthogonal Real Chains vs. Single Complex Chain**: Comparison between two independent real Fourier chains (size $N_x + N_y = 2N$) vs. a single complex Fourier chain $z(t) = \sum c_k e^{i k t}$ at equal total circle count.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

# Ensure project root is available
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.outline import get_india_contour
from src.fourier import compute_coefficients, reconstruct_2d, compute_contour_error, sort_harmonics_by_amplitude

# Aesthetic styling for publication
plt.style.use('dark_background')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['figure.facecolor'] = '#05070d'
plt.rcParams['axes.facecolor'] = '#0a0f1d'
plt.rcParams['axes.edgecolor'] = '#1f293d'
plt.rcParams['grid.color'] = '#162035'

## 1. Load Ground Truth Outline & Compute Fourier Series

In [ ]:
geojson_path = PROJECT_ROOT / "data" / "india_outline.geojson"
M = 8192
x_true, y_true, t_true = get_india_contour(mode="geojson", filepath=geojson_path, M=M)

# Compute maximum available Fourier harmonics
(a0, c0), a, b, c, d = compute_coefficients(x_true, y_true, N_harmonics=2000)
order_x, order_y = sort_harmonics_by_amplitude(a, b, c, d)

print(f"Loaded M = {M} arc-length resampled points.")
print(f"Computed N = 2000 Fourier coefficients.")

## 2. Quantitative Error vs. N (Log-Log Scale from N = 1 to 2000)

We evaluate the **mean nearest-neighbor distance** normalized by bounding-box diagonal (expressed as a percentage) and the **symmetric Hausdorff distance**.

In [ ]:
eval_N = np.unique(np.logspace(0, np.log10(2000), 45).astype(int))
mean_errors = []
hausdorff_errors = []

for n in eval_N:
    xr, yr = reconstruct_2d(t_true, (a0, c0), a, b, c, d, N=n, sort_by_amplitude=True)
    err = compute_contour_error(x_true, y_true, xr, yr)
    mean_errors.append(err["mean_error_pct"])
    hausdorff_errors.append(err["hausdorff_pct"])

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.loglog(eval_N, mean_errors, marker='o', markersize=4, color='#00d2d3', label='Mean Error (% diagonal)')
ax.loglog(eval_N, hausdorff_errors, marker='s', markersize=4, color='#ff6b6b', label='Hausdorff Error (% diagonal)')

# Highlight N = 1000 benchmark
idx_1000 = np.argmin(np.abs(eval_N - 1000))
ax.scatter([eval_N[idx_1000]], [mean_errors[idx_1000]], color='#f5a623', s=120, zorder=5)
ax.annotate(f"N=1000: error = {mean_errors[idx_1000]:.2f}%", 
            (eval_N[idx_1000], mean_errors[idx_1000]),
            textcoords="offset points", xytext=(-40, 15), color='#f5a623', weight='bold')

ax.set_title("Fourier Reconstruction Error vs. Harmonic Order N (Log-Log)", fontsize=14, color='#ffffff', pad=12)
ax.set_xlabel("Number of Harmonics N", fontsize=12, color='#94a3b8')
ax.set_ylabel("Error (% of Bounding Box Diagonal)", fontsize=12, color='#94a3b8')
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.legend(frameon=True, facecolor='#0e1726', edgecolor='#1f293d')
plt.tight_layout()
plt.show()

## 3. Side-by-Side Visual Reconstructions: N = 1, 5, 20, 100, 1000

In [ ]:
sample_Ns = [1, 5, 20, 100, 1000]
fig, axes = plt.subplots(1, 5, figsize=(20, 5), dpi=150)

for ax, n in zip(axes, sample_Ns):
    xr, yr = reconstruct_2d(t_true, (a0, c0), a, b, c, d, N=n, sort_by_amplitude=True)
    err = compute_contour_error(x_true, y_true, xr, yr)["mean_error_pct"]
    
    # Faint ground truth
    ax.fill(x_true, y_true, color='#0b1220', alpha=0.9)
    ax.plot(x_true, y_true, color='#162035', linewidth=0.8)
    
    # Reconstructed contour
    ax.plot(xr, yr, color='#ff6b6b', linewidth=1.5)
    
    ax.set_title(f"N = {n}\nerror = {err:.2f}%", fontsize=12, color='#ffffff')
    ax.set_aspect('equal')
    ax.axis('off')

plt.suptitle("Orthogonal Fourier Evolution Across Harmonic Orders", fontsize=15, color='#00d2d3', y=1.03)
plt.tight_layout()
plt.show()

## 4. Orthogonal Real Epicycles vs. Single Complex Fourier Chain

### Theoretical Comparison
- **Single Complex Chain (Standard Epicycles)**:
  In the standard complex formulation, $z(t) = x(t) + i y(t)$ is decomposed into:
  $$z(t) = \sum_{k=-K}^K C_k e^{i k t}$$
  Here, each term represents a 2D rotating vector in the complex plane. A chain of $K_{total} = 2K$ rotating circles produces the $(x, y)$ coordinate directly at the tip of the single chain.

- **Orthogonal Real Chains (Our Formulation)**:
  Two separate 1D real Fourier series are used:
  $$x(t) = a_0 + \sum_{k=1}^{N_x} R_{x,k} \cos(kt - \phi_{x,k})$$
  $$y(t) = c_0 + \sum_{k=1}^{N_y} R_{y,k} \cos(kt - \psi_{y,k})$$
  Total circle count: $N_{total} = N_x + N_y = 2N$ when $N_x = N_y = N$.

### Is there an accuracy difference at equal total circle count?
Let us test both at equal total circle counts $C = 20, 50, 100, 200, 500, 1000$.

In [ ]:
# Complex FFT
z = x_true + 1j * y_true
Z = np.fft.fft(z) / M
freqs = np.fft.fftfreq(M, d=1.0/M)

# Sort complex harmonics by magnitude
complex_magnitudes = np.abs(Z)
complex_order = np.argsort(-complex_magnitudes)
# Exclude DC term (freq = 0)
complex_order_ac = [idx for idx in complex_order if freqs[idx] != 0]

def reconstruct_complex(K_circles):
    z_rec = np.full_like(z, Z[0])
    for i in range(K_circles):
        idx = complex_order_ac[i]
        k = freqs[idx]
        z_rec += Z[idx] * np.exp(1j * k * t_true)
    return np.real(z_rec), np.imag(z_rec)

total_circle_counts = [10, 20, 50, 100, 200, 500, 1000]
ortho_errors = []
complex_errors = []

for total_c in total_circle_counts:
    # Orthogonal: split circles equally between X and Y chains
    N_each = total_c // 2
    xr_o, yr_o = reconstruct_2d(t_true, (a0, c0), a, b, c, d, N=N_each, sort_by_amplitude=True)
    err_o = compute_contour_error(x_true, y_true, xr_o, yr_o)["mean_error_pct"]
    ortho_errors.append(err_o)
    
    # Complex: K_circles = total_c
    xr_c, yr_c = reconstruct_complex(total_c)
    err_c = compute_contour_error(x_true, y_true, xr_c, yr_c)["mean_error_pct"]
    complex_errors.append(err_c)

fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
ax.plot(total_circle_counts, ortho_errors, marker='o', color='#00d2d3', label='Orthogonal Real Chains (X + Y = Total)')
ax.plot(total_circle_counts, complex_errors, marker='s', color='#f5a623', linestyle='--', label='Single Complex Chain (Total Circles)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title("Orthogonal vs. Complex Fourier at Equal Total Epicycle Count", fontsize=13, color='#ffffff')
ax.set_xlabel("Total Rotating Circles (Epicycles)", fontsize=11, color='#94a3b8')
ax.set_ylabel("Mean Error (% Bounding Box Diagonal)", fontsize=11, color='#94a3b8')
ax.grid(True, which='both', linestyle=':', alpha=0.6)
ax.legend(facecolor='#0e1726', edgecolor='#1f293d')
plt.tight_layout()
plt.show()

### Findings & Discussion:
1. **Mathematical Equivalence in the Limit**: As $N \to \infty$, both representations converge uniformly to the true contour with error decaying as $\mathcal{O}(N^{-1})$ for continuous piecewise-smooth curves.
2. **Harmonic Allocation**: In the complex Fourier formulation, positive and negative frequencies $(+k, -k)$ decouple counter-rotating elliptical components. In the orthogonal real formulation, harmonics are decoupled along the coordinate axes $(x, y)$, allowing independent allocation of degrees of freedom.
3. **Visual & Pedagogical Superiority**: The orthogonal formulation allows direct visualization of how 1D mechanical harmonic motion in $x$ and $y$ independently synthesizes a 2D closed spatial geometry via Cartesian intersection lines.